# Практика 3. Приведення до tidy data

Мета: перетворити таблицю з wide-формату в tidy (long) і назад, класифікувати змінні за шкалою вимірювання та прочитати й записати таблицю у форматах CSV і JSON.

**Інструкція:** у першій Python-клітинці змініть `city`, якщо ваш номер варіанта відповідає іншому місту. За замовчуванням обрано Київ, варіант 1.

In [1]:
from pathlib import Path
import pandas as pd

variants = {
    'Київ': [78, 74, 68, 58, 50, 44, 40, 42, 52, 66, 76, 82],
    'Львів': [82, 78, 72, 62, 55, 48, 44, 46, 56, 70, 80, 85],
    'Одеса': [70, 66, 60, 50, 42, 36, 32, 34, 44, 58, 68, 74],
    'Харків': [76, 72, 65, 55, 46, 40, 36, 38, 48, 62, 74, 80],
    'Дніпро': [74, 70, 63, 53, 45, 39, 35, 37, 47, 60, 72, 78],
    'Чернігів': [80, 76, 70, 60, 52, 46, 42, 44, 54, 68, 78, 84],
    'Івано-Франківськ': [84, 80, 74, 64, 57, 50, 46, 48, 58, 72, 82, 86],
    'Полтава': [77, 73, 66, 56, 47, 41, 37, 39, 49, 63, 75, 81],
    'Суми': [79, 75, 69, 59, 51, 45, 41, 43, 53, 67, 77, 83],
    'Ужгород': [72, 68, 61, 51, 44, 38, 34, 36, 46, 60, 70, 76]
}
months = ['Січ', 'Лют', 'Бер', 'Кві', 'Тра', 'Чер', 'Лип', 'Сер', 'Вер', 'Жов', 'Лис', 'Гру']
city = 'Київ'
if city not in variants:
    raise ValueError(f'Невідоме місто: {city}. Оберіть одне з: {list(variants)}')

wide = pd.DataFrame([[city, *variants[city]]], columns=['місто', *months])
print(f'Варіант {list(variants).index(city) + 1}: {city}')
display(wide)

Варіант 1: Київ


,місто,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру
0,Київ,78,74,68,58,50,44,40,42,52,66,76,82


## Завдання 1. Побудова wide-таблиці

`wide` містить один рядок для міста, а кожен місяць записаний окремим стовпцем. Це саме той формат, у якому подано варіант завдання.

## Завдання 2. Приведення до tidy (`melt`)

In [2]:
tidy = wide.melt(
    id_vars='місто',
    var_name='місяць',
    value_name='хмарність'
)
print(f'Форма wide: {wide.shape}; форма tidy: {tidy.shape}')
display(tidy)
assert list(tidy.columns) == ['місто', 'місяць', 'хмарність']
assert tidy.shape == (12, 3)

Форма wide: (1, 13); форма tidy: (12, 3)


,місто,місяць,хмарність
0,Київ,Січ,78
1,Київ,Лют,74
2,Київ,Бер,68
3,Київ,Кві,58
4,Київ,Тра,50
5,Київ,Чер,44
6,Київ,Лип,40
7,Київ,Сер,42
8,Київ,Вер,52
9,Київ,Жов,66


Після `melt()` отримано 12 рядків, тому що кожна пара «місто–місяць» є окремим спостереженням. У wide-таблиці 12 місяців були розкладені по стовпцях, а в tidy-таблиці назва місяця стала значенням одного стовпця.

## Завдання 3. Класифікація змінних

- **`місто` — номінальна шкала:** це назва категорії без природного порядку.
- **`місяць` — порядкова шкала:** місяці мають природний календарний порядок, але назви місяців не є кількісними вимірюваннями.
- **`хмарність` — шкала відношень:** відсоток має природний нуль, рівні різниці та змістовне порівняння відношень, наприклад 80% удвічі більше за 40%.

## Завдання 4. Зворотне перетворення й перевірка

In [3]:
back = tidy.pivot(index='місто', columns='місяць', values='хмарність').reset_index()
back = back[wide.columns]
display(back)
same_values = wide.reset_index(drop=True).equals(back.reset_index(drop=True))
print(f'Значення wide і результату pivot збігаються: {same_values}')
assert same_values

місяць,місто,Січ,Лют,Бер,Кві,Тра,Чер,Лип,Сер,Вер,Жов,Лис,Гру
0,Київ,78,74,68,58,50,44,40,42,52,66,76,82


Значення wide і результату pivot збігаються: True


`pivot()` повертає назви місяців у стовпці й відновлює один рядок міста. Отримана таблиця має ті самі значення, що й початкова wide-таблиця; перевірка `equals()` підтверджує це.

## Завдання 5. Читання й запис CSV та JSON

In [4]:
output_dir = Path.cwd()
csv_path = output_dir / 'variant.csv'
json_path = output_dir / 'variant.json'
tidy.to_csv(csv_path, index=False, encoding='utf-8')
tidy.to_json(json_path, orient='records', force_ascii=False, indent=2)

reloaded_csv = pd.read_csv(csv_path)
reloaded_json = pd.read_json(json_path, orient='records')
print(f'Створено: {csv_path.name}, {json_path.name}')
print('CSV:')
print(csv_path.read_text(encoding='utf-8'))
print('JSON:')
print(json_path.read_text(encoding='utf-8'))

assert reloaded_csv.equals(tidy)
assert reloaded_json.equals(tidy)
print('Перевірка: CSV і JSON після читання збігаються з tidy-таблицею.')

Створено: variant.csv, variant.json
CSV:
місто,місяць,хмарність
Київ,Січ,78
Київ,Лют,74
Київ,Бер,68
Київ,Кві,58
Київ,Тра,50
Київ,Чер,44
Київ,Лип,40
Київ,Сер,42
Київ,Вер,52
Київ,Жов,66
Київ,Лис,76
Київ,Гру,82

JSON:
[
  {
    "місто":"Київ",
    "місяць":"Січ",
    "хмарність":78
  },
  {
    "місто":"Київ",
    "місяць":"Лют",
    "хмарність":74
  },
  {
    "місто":"Київ",
    "місяць":"Бер",
    "хмарність":68
  },
  {
    "місто":"Київ",
    "місяць":"Кві",
    "хмарність":58
  },
  {
    "місто":"Київ",
    "місяць":"Тра",
    "хмарність":50
  },
  {
    "місто":"Київ",
    "місяць":"Чер",
    "хмарність":44
  },
  {
    "місто":"Київ",
    "місяць":"Лип",
    "хмарність":40
  },
  {
    "місто":"Київ",
    "місяць":"Сер",
    "хмарність":42
  },
  {
    "місто":"Київ",
    "місяць":"Вер",
    "хмарність":52
  },
  {
    "місто":"Київ",
    "місяць":"Жов",
    "хмарність":66
  },
  {
    "місто":"Київ",
    "місяць":"Лис",
    "хмарність":76
  },
  {
    "місто":"Київ",
    "місяць

CSV — це простий табличний текст: рядки відповідають записам, а значення розділені комами. JSON зберігає записи як масив об'єктів із парами «ключ–значення», тому краще передає вкладену структуру та назви полів, але зазвичай займає більше тексту.

## Короткі відповіді на питання для здачі

1. **Чому `місто` — номінальна шкала, а `хмарність` — шкала відношень?** Назви міст лише розрізняють категорії й не мають числового порядку. Хмарність у відсотках має нульове значення та змістовні відношення між величинами.
2. **Чому wide-таблиця не є tidy?** У tidy-даних кожна змінна має окремий стовпець, кожне спостереження — рядок, а кожне значення — клітинку. У wide-таблиці змінна «місяць» закодована в назвах 12 стовпців; після `melt()` вона стає одним стовпцем, а місячні спостереження — окремими рядками.
3. **Чим CSV відрізняється від JSON?** CSV орієнтований на прямокутну таблицю з роздільником, а JSON — на структуровані об'єкти та масиви. Для простої таблиці обидва підходять, але JSON краще зберігає вкладені й неоднорідні структури.

## Підсумок

Wide-таблицю власного варіанта перетворено на tidy через `melt()`, відновлено через `pivot()`, класифіковано три змінні за шкалами вимірювання та перевірено запис і читання даних у CSV і JSON.